<h1>Extracting Data from Flight Labs API</h1>

In [1]:
import os
from dotenv import load_dotenv
from utils import extract, transform, load
import psycopg2
from sqlalchemy import create_engine

load_dotenv()
access_key = os.getenv('ACCESS_KEY')

# Define API endpoint
url = 'https://www.goflightlabs.com/flights'

# Extracting data from API endpoint
flight_data_raw = extract(url, access_key)
print(flight_data_raw.shape)

Returned status code: 200
Extraction complete
(100, 22)


<H1>Exploring the Raw Data </h1>

In [2]:
# EDA
display(flight_data_raw.head(), flight_data_raw.info(), flight_data_raw.describe())

print('\nNumber of null values for every column feature\n')
flight_data_raw.isnull().sum()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 22 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   hex            100 non-null    object 
 1   reg_number     100 non-null    object 
 2   flag           100 non-null    object 
 3   lat            100 non-null    float64
 4   lng            100 non-null    float64
 5   alt            100 non-null    int64  
 6   dir            100 non-null    float64
 7   speed          100 non-null    int64  
 8   v_speed        100 non-null    int64  
 9   flight_number  100 non-null    object 
 10  flight_icao    100 non-null    object 
 11  flight_iata    100 non-null    object 
 12  dep_icao       100 non-null    object 
 13  dep_iata       100 non-null    object 
 14  arr_icao       100 non-null    object 
 15  arr_iata       100 non-null    object 
 16  airline_icao   100 non-null    object 
 17  airline_iata   100 non-null    object 
 18  aircraft_ic

,hex,reg_number,flag,lat,lng,alt,dir,speed,v_speed,flight_number,...,dep_icao,dep_iata,arr_icao,arr_iata,airline_icao,airline_iata,aircraft_icao,updated,status,type
0,789280,B-KKD,HK,22.763867,118.856033,11389,59.5,925,0,686,...,VHHH,HKG,RJBB,KIX,HKE,UO,A21N,1764725056,en-route,adsb
1,758531,RP-C4112,PH,13.197088,121.657410,7236,327.0,794,0,382,...,RPMY,CGY,RPLL,MNL,CEB,5J,A321,1764725056,en-route,adsb
2,801604,VT-BXE,IN,22.131725,61.424461,11770,108.9,955,0,852,...,OKKK,KWI,VOML,IXE,AXB,IX,B38M,1764725056,en-route,adsb
3,7C4798,VH-OFE,AU,-23.300169,146.291415,10688,182.1,894,0,33,...,YBCS,CNS,YMML,MEL,JST,JQ,A21N,1764725056,en-route,adsb
4,E80620,CC-DIC,CL,-32.256702,-62.075474,11450,302.0,765,0,3108,...,SABE,AEP,SACO,COR,JES,WJ,A21N,1764725056,en-route,adsb


None

,lat,lng,alt,dir,speed,v_speed,updated
count,100.000000,100.000000,100.000000,100.000000,100.000000,100.0,1.000000e+02
mean,19.814942,14.474250,8473.410000,168.513000,747.260000,0.0,1.764725e+09
std,23.621079,102.591497,3538.364086,102.909744,199.930625,0.0,0.000000e+00
min,-34.903688,-157.185125,376.000000,4.500000,275.000000,0.0,1.764725e+09
25%,11.144106,-92.573600,5903.750000,80.450000,630.750000,0.0,1.764725e+09
50%,23.173334,59.463889,10108.500000,143.900000,775.500000,0.0,1.764725e+09
75%,37.426738,108.455721,11316.250000,276.325000,881.000000,0.0,1.764725e+09
max,60.669412,147.629819,12516.000000,354.400000,1179.000000,0.0,1.764725e+09



Number of null values for every column feature



hex              0
reg_number       0
flag             0
lat              0
lng              0
alt              0
dir              0
speed            0
v_speed          0
flight_number    0
flight_icao      0
flight_iata      0
dep_icao         0
dep_iata         0
arr_icao         0
arr_iata         0
airline_icao     0
airline_iata     0
aircraft_icao    0
updated          0
status           0
type             0
dtype: int64

<h1>Data Cleaning</h1>

<li>Replacing missing values in "squawk" column with "unknown" if the column is pulled during extraction</li>
<li>Replacing missing values in 'alt', 'speed' and v_speed to 0</li>
<li>Renaming columns</li>
<li>Converting "updated" values to datetime</li>


In [3]:
flight_data_clean = transform(flight_data_raw)

display(flight_data_clean.head())

print('\nNumber of null values for every column feature\n')
flight_data_clean.isnull().sum()


transform complete


,hex,reg_number,flag,latitude,longitude,altitude_ft,dir,speed_mph,v_speed_mph,flight_number,...,departure_icao,departure_iata,arrival_icao,arrival_iata,airline_icao,airline_iata,aircraft_icao,updated,status,type
0,789280,B-KKD,HK,22.763867,118.856033,37365.48676,59.5,574.768175,0.0,686,...,VHHH,HKG,RJBB,KIX,HKE,UO,A21N,2025-12-02 20:24:16,en-route,adsb
1,758531,RP-C4112,PH,13.197088,121.657410,23740.15824,327.0,493.368574,0.0,382,...,RPMY,CGY,RPLL,MNL,CEB,5J,A321,2025-12-02 20:24:16,en-route,adsb
2,801604,VT-BXE,IN,22.131725,61.424461,38615.48680,108.9,593.409305,0.0,852,...,OKKK,KWI,VOML,IXE,AXB,IX,B38M,2025-12-02 20:24:16,en-route,adsb
3,7C4798,VH-OFE,AU,-23.300169,146.291415,35065.61792,182.1,555.505674,0.0,33,...,YBCS,CNS,YMML,MEL,JST,JQ,A21N,2025-12-02 20:24:16,en-route,adsb
4,E80620,CC-DIC,CL,-32.256702,-62.075474,37565.61800,302.0,475.348815,0.0,3108,...,SABE,AEP,SACO,COR,JES,WJ,A21N,2025-12-02 20:24:16,en-route,adsb



Number of null values for every column feature



hex               0
reg_number        0
flag              0
latitude          0
longitude         0
altitude_ft       0
dir               0
speed_mph         0
v_speed_mph       0
flight_number     0
flight_icao       0
flight_iata       0
departure_icao    0
departure_iata    0
arrival_icao      0
arrival_iata      0
airline_icao      0
airline_iata      0
aircraft_icao     0
updated           0
status            0
type              0
dtype: int64

<h1>Loading Cleaned Data to CSVs and Postgres </h1>

In [ ]:
def load_to_csv(df_raw, df_clean):
    # Create data folder if it doesn't exist
    os.makedirs('data', exist_ok=True)

    df_raw.to_csv('data/flights_data_raw.csv', index=False)
    df_clean.to_csv('data/flights_data_clean.csv')


def load_to_postgres(df_raw, df_clean, conn):
    df_raw.to_sql(
        name='flights_realtime_raw',
        con=conn,
        if_exists='append',
        index=False
    )

    df_clean.to_sql(
        name='flights_realtime_clean',
        con=conn,
        if_exists='append',
        index=False
    )
    

    print('load complete')

In [ ]:
# Connecting to local flight data database
dbname=os.getenv('DB_NAME')
user=os.getenv('DB_USER')
password=os.getenv('DB_PASSWORD')
host=os.getenv('DB_HOST')
port=os.getenv('DB_PORT')

conn = create_engine(f'postgresql+psycopg2://{user}:{password}@{host}:{port}/{dbname}')

# Loading raw and clean data
load_to_csv(flight_data_raw, flight_data_clean)
load_to_postgres(flight_data_raw, flight_data_clean, conn)

load complete
